# ComfyUI Google Cloud TPU v5e-8 Qwen-Image 2.1 Diffusion Pipeline

This notebook executes native Google Cloud TPU support in ComfyUI (torch.device('tpu') via torch_tpu), running on a Kaggle Google Cloud TPU v5e-8 (TpuV5E8) accelerator, executing the complete text-to-image diffusion pipeline for Qwen-Image 2.1.

In [ ]:
import os
import sys
import subprocess

OUTPUT_DIR = "/kaggle/working/output"
os.makedirs(OUTPUT_DIR, exist_ok=True)
LOG_FILE = os.path.join(OUTPUT_DIR, "execution.log")

def log(msg):
    print(msg, flush=True)
    with open(LOG_FILE, "a", encoding="utf-8") as f:
        print(msg, file=f, flush=True)

log(f"Python Version: {sys.version}")

# Check existing TPU environment variables
log("=== Existing TPU / Accel Environment Variables ===")
for k, v in sorted(os.environ.items()):
    if any(x in k.lower() for x in ["tpu", "pjrt", "xla", "jax"]):
        log(f"  {k} = {v}")

# Configure TPU v5e environment variables
os.environ["TPU_SKIP_MDS_QUERY"] = "1"
os.environ["PJRT_DEVICE"] = "TPU"
os.environ.setdefault("TPU_ACCELERATOR_TYPE", "v5e-8")
os.environ.setdefault("TPU_CHIPS_PER_HOST_BOUNDS", "2,2,1")
os.environ.setdefault("TPU_HOST_BOUNDS", "1,1,1")
os.environ.setdefault("TPU_WORKER_ID", "0")
os.environ.setdefault("TPU_RUNTIME_METRICS_PORTS", "8431,8432,8433,8434")
os.environ.setdefault("TPU_WORKER_HOSTNAMES", "localhost")

# Check /dev/accel*
res = subprocess.run(["ls", "-l", "/dev/accel*"], capture_output=True, text=True)
log('Hardware devices (/dev/accel*):')
log(res.stdout or res.stderr)


In [ ]:
# Upgrade PyTorch to 2.14+ (CPU wheel) and install torch_tpu + gguf
log("Installing PyTorch CPU and torch_tpu...")
res = subprocess.run([
    sys.executable, "-m", "pip", "install", "-U",
    "torch", "torchvision", "--index-url", "https://download.pytorch.org/whl/cpu"
], capture_output=True, text=True)
log(f"pip install torch returncode: {res.returncode}")

res2 = subprocess.run([
    sys.executable, "-m", "pip", "install", "--no-cache-dir", "torch_tpu", "gguf"
], capture_output=True, text=True)
log(f"pip install torch_tpu returncode: {res2.returncode}")

import torch
log(f"PyTorch Version: {torch.__version__}")

try:
    import torch_tpu
    log(f"torch_tpu imported successfully: {torch_tpu}")
except Exception as e:
    log(f"Error importing torch_tpu: {e}")

# Test native TPU device
try:
    dev = torch.device("tpu")
    x = torch.randn((4, 4), device=dev)
    log(f"Native torch.device('tpu') created: {x.device}")
    y = x @ x
    log(f"Native TPU matmul verified: shape={y.shape}")
except Exception as e:
    log(f"Notice on native torch.device('tpu') test: {e}")

has_acc = hasattr(torch, "accelerator") and torch.accelerator.is_available()
log(f"torch.accelerator available: {has_acc}")
if has_acc:
    log(f"Current accelerator: {torch.accelerator.current_accelerator()}")
    log(f"Accelerator device count: {torch.accelerator.device_count()}")
    try:
        free_b, total_b = torch.accelerator.get_memory_info()
        log(f"TPU Memory: free={free_b / 1e9:.2f} GB, total={total_b / 1e9:.2f} GB")
    except Exception as e:
        log(f"Memory info: {e}")


In [ ]:
# Clone ComfyUI repository & Setup ComfyUI-GGUF with Patches
import shutil
COMFY_DIR = "/tmp/ComfyUI"
if os.path.exists(COMFY_DIR):
    shutil.rmtree(COMFY_DIR, ignore_errors=True)

subprocess.run(["git", "clone", "--depth", "1", "-b", "feat/tpu-multi-core-support",
                "https://github.com/vuthanhtrung2010/ComfyUI.git", COMFY_DIR], check=True)
log(f"Cloned ComfyUI to {COMFY_DIR}")

# Install ComfyUI dependencies (excluding torch/torchvision)
with open(f"{COMFY_DIR}/requirements.txt", "r") as f:
    reqs = [l.strip() for l in f if l.strip() and not l.startswith(("torch", "torchvision"))]
with open("/tmp/reqs.txt", "w") as f:
    for r in reqs:
        print(r, file=f)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "/tmp/reqs.txt"], check=True)
log("ComfyUI core requirements installed.")

# Install ComfyUI-GGUF custom node
gguf_node_dir = os.path.join(COMFY_DIR, "custom_nodes", "ComfyUI-GGUF")
subprocess.run(["git", "clone", "https://github.com/city96/ComfyUI-GGUF.git", gguf_node_dir], check=True)

loader_py = os.path.join(gguf_node_dir, "loader.py")
with open(loader_py, "r", encoding="utf-8") as f:
    src = f.read()
src = src.replace('"qwen_image"', '"qwen_image", "qwen_image21"')
src = src.replace('raise ValueError(f"This model is not currently supported - ({e})")', 'arch_str = "qwen_image"')
with open(loader_py, "w", encoding="utf-8") as f:
    f.write(src)
log("Successfully patched ComfyUI-GGUF for Qwen-Image 2.1!")


In [ ]:
# Download Qwen-Image 2.1 Model Checkpoints using Python Subprocess (resilient)
def download_file(url, dest):
    os.makedirs(os.path.dirname(dest), exist_ok=True)
    if os.path.exists(dest) and os.path.getsize(dest) > 1024 * 1024:
        log(f"[SKIP] Already present: {dest} ({os.path.getsize(dest) / 1e6:.1f} MB)")
        return
    log(f"[DOWNLOAD] {dest} from {url}...")
    subprocess.run(["curl", "-L", "-C", "-", "--retry", "5", "-o", dest, url], check=True)
    log(f"[DONE] {dest} ({os.path.getsize(dest) / 1e6:.1f} MB)")

models = [
    ("models/unet/qwen-image-2.1-UC-Q4_K_M.gguf",
     "https://huggingface.co/abenzerps/Qwen-Image-2.1-Turbo-Quantized/resolve/main/qwen-image-2.1-turbo-Q4_K_M.gguf"),
    ("models/clip/qwen3vl_8b_int8_convrot.safetensors",
     "https://huggingface.co/abenzerps/Qwen-Image-2.1-Uncensored-Quantized/resolve/main/text_encoders/qwen3vl_8b_int8_convrot.safetensors"),
    ("models/vae/qwen_image_2.1_vae_bf16.safetensors",
     "https://huggingface.co/abenzerps/Qwen-Image-2.1-Uncensored-Quantized/resolve/main/vae/qwen_image_2.1_vae_bf16.safetensors")
]

for rel, url in models:
    dest = os.path.join(COMFY_DIR, rel)
    download_file(url, dest)

log("All models verified on disk!")


In [ ]:
# Launch ComfyUI Server on TPU Core 0 and Execute Qwen-Image 2.1 Pipeline
import subprocess
import time
import urllib.request
import urllib.error
import json

os.chdir(COMFY_DIR)
log_server = open(os.path.join(OUTPUT_DIR, "comfyui_server.log"), "w")
server_proc = subprocess.Popen(
    [sys.executable, "main.py", "--tpu-device", "0", "--port", "8188", "--listen", "127.0.0.1"],
    stdout=log_server,
    stderr=subprocess.STDOUT,
    cwd=COMFY_DIR
)
log(f"ComfyUI server started with PID: {server_proc.pid}")

server_ready = False
for _ in range(60):
    time.sleep(1)
    try:
        req = urllib.request.urlopen("http://127.0.0.1:8188/system_stats", timeout=2)
        if req.status == 200:
            stats = json.loads(req.read().decode())
            log(f"Server is online! Detected devices: {[d["name"] for d in stats.get("devices", [])]}")
            server_ready = True
            break
    except Exception:
        pass

if not server_ready:
    log("Server failed to come online within 60s. Check comfyui_server.log")
    raise RuntimeError("ComfyUI server timeout")

# Neptune Motion Qwen-Image 2.1 Workflow
workflow = {
    "1": {"class_type": "UnetLoaderGGUF", "inputs": {"unet_name": "qwen-image-2.1-UC-Q4_K_M.gguf"}},
    "2": {"class_type": "CLIPLoader", "inputs": {"clip_name": "qwen3vl_8b_int8_convrot.safetensors", "type": "qwen_image", "device": "default"}},
    "3": {"class_type": "VAELoader", "inputs": {"vae_name": "qwen_image_2.1_vae_bf16.safetensors"}},
    "4": {"class_type": "TextEncodeQwenImage21", "inputs": {
        "clip": ["2", 0],
        "prompt": "An artistic educational diagram explaining Shannon information theory and entropy, detailed whiteboard drawing",
        "negative_prompt": "blurry, low quality, distorted, artifacts",
        "resolution": 1024
    }},
    "5": {"class_type": "EmptyLatentImage", "inputs": {"width": 1024, "height": 640, "batch_size": 1}},
    "6": {"class_type": "KSampler", "inputs": {
        "model": ["1", 0],
        "seed": 42,
        "steps": 8,
        "cfg": 1.0,
        "sampler_name": "euler",
        "scheduler": "simple",
        "positive": ["4", 0],
        "negative": ["4", 1],
        "latent_image": ["5", 0],
        "denoise": 1.0
    }},
    "7": {"class_type": "VAEDecode", "inputs": {"samples": ["6", 0], "vae": ["3", 0]}},
    "8": {"class_type": "SaveImage", "inputs": {"images": ["7", 0], "filename_prefix": "explainer"}}
}

log("Dispatching Neptune Motion Qwen-Image 2.1 text-to-image workflow...")
data = json.dumps({"prompt": workflow, "client_id": "kaggle_tpu_run"}).encode("utf-8")
req = urllib.request.Request("http://127.0.0.1:8188/prompt", data=data, headers={"Content-Type": "application/json"})
resp = urllib.request.urlopen(req)
prompt_res = json.loads(resp.read().decode())
prompt_id = prompt_res["prompt_id"]
log(f"Prompt successfully queued with ID: {prompt_id}")

start_time = time.time()
completed = False
while time.time() - start_time < 900:
    time.sleep(3)
    try:
        h_req = urllib.request.urlopen("http://127.0.0.1:8188/history")
        history = json.loads(h_req.read().decode())
        if prompt_id in history:
            entry = history[prompt_id]
            status = entry.get("status", {})
            if status.get("completed", False):
                elapsed = time.time() - start_time
                log(f"[SUCCESS] Qwen-Image 2.1 completed in {elapsed:.1f}s!")
                log(f"Outputs: {entry.get('outputs', {})}")
                completed = True
                break
            elif status.get("status_str") == "error":
                log(f"[ERROR] Workflow execution failed: {status}")
                break
    except Exception:
        pass

# Copy generated images to /kaggle/working/output
comfy_out = os.path.join(COMFY_DIR, "output")
if os.path.exists(comfy_out):
    for f in os.listdir(comfy_out):
        if f.endswith((".png", ".jpg", ".latent")):
            shutil.copy(os.path.join(comfy_out, f), os.path.join(OUTPUT_DIR, f))
            log(f"Copied output: {f}")

server_proc.terminate()
server_proc.wait(timeout=10)
log("Server terminated cleanly.")
log(f"Generated artifacts in {OUTPUT_DIR}:")
res = subprocess.run(["ls", "-lh", OUTPUT_DIR], capture_output=True, text=True)
log(res.stdout)
